In [5]:
from pathlib import Path
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
base_dir = Path("data/UCI_HAR_Dataset")
base_dir.mkdir(parents=True, exist_ok=True)

url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
zip_path = base_dir / "UCI_HAR.zip"
extract_dir = base_dir

target_file = extract_dir / "UCI HAR Dataset" / "train" / "X_train.txt"

if not target_file.exists():
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

In [ ]:
candidate_dirs = [
    Path("data/UCI_HAR_Dataset/UCI HAR Dataset"),
    Path("UCI_HAR_Dataset/UCI HAR Dataset"),
    Path("UCI HAR Dataset"),
]

har_dir = next((p for p in candidate_dirs if (p / "features.txt").exists()), None)
if har_dir is None:
    raise FileNotFoundError(
        "Could not find 'UCI HAR Dataset'. Run download cell first or check dataset path."
    )

features = pd.read_csv(
    har_dir / "features.txt",
    sep=r"\s+",
    header=None,
    names=["feature_idx", "feature_name"]
)
activity_labels = pd.read_csv(
    har_dir / "activity_labels.txt",
    sep=r"\s+",
    header=None,
    names=["activity_id", "activity"]
)

clean_feature_names = (
    features["feature_name"]
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)

name_counts = {}
unique_feature_names = []
for name in clean_feature_names:
    count = name_counts.get(name, 0)
    unique_name = name if count == 0 else f"{name}_{count}"
    unique_feature_names.append(unique_name)
    name_counts[name] = count + 1

X_train = pd.read_csv(har_dir / "train" / "X_train.txt", sep=r"\s+", header=None)
y_train = pd.read_csv(har_dir / "train" / "y_train.txt", sep=r"\s+", header=None, names=["activity_id"])
subject_train = pd.read_csv(har_dir / "train" / "subject_train.txt", sep=r"\s+", header=None, names=["subject"])

X_test = pd.read_csv(har_dir / "test" / "X_test.txt", sep=r"\s+", header=None)
y_test = pd.read_csv(har_dir / "test" / "y_test.txt", sep=r"\s+", header=None, names=["activity_id"])
subject_test = pd.read_csv(har_dir / "test" / "subject_test.txt", sep=r"\s+", header=None, names=["subject"])

X_train.columns = unique_feature_names
X_test.columns = unique_feature_names

train_df = pd.concat([subject_train, y_train, X_train], axis=1)
test_df = pd.concat([subject_test, y_test, X_test], axis=1)
df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

a_map = dict(zip(activity_labels["activity_id"], activity_labels["activity"]))
df["activity"] = df["activity_id"].map(a_map)

print(df.columns.tolist())

df.head()